<a href="https://colab.research.google.com/github/joyngeno21-ui/Joy-AI-Assignment/blob/main/Notebooks/Chap13/13_4_Graph_Attention_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 13.4: Graph attention networks**

This notebook builds a graph attention mechanism from scratch, as discussed in section 13.8.6 of the book and illustrated in figure 13.12c

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.



In [1]:
import numpy as np
import matplotlib.pyplot as plt

The self-attention mechanism maps $N$ inputs $\mathbf{x}_{n}\in\mathbb{R}^{D}$ and returns $N$ outputs $\mathbf{x}'_{n}\in \mathbb{R}^{D}$.  



In [2]:
# Set seed so we get the same random numbers
np.random.seed(1)
# Number of nodes in the graph
N = 8
# Number of dimensions of each input
D = 4

# Define a graph
A = np.array([[0,1,0,1,0,0,0,0],
              [1,0,1,1,1,0,0,0],
              [0,1,0,0,1,0,0,0],
              [1,1,0,0,1,0,0,0],
              [0,1,1,1,0,1,0,1],
              [0,0,0,0,1,0,1,1],
              [0,0,0,0,0,1,0,0],
              [0,0,0,0,1,1,0,0]]);
print(A)

# Let's also define some random data
X = np.random.normal(size=(D,N))

[[0 1 0 1 0 0 0 0]
 [1 0 1 1 1 0 0 0]
 [0 1 0 0 1 0 0 0]
 [1 1 0 0 1 0 0 0]
 [0 1 1 1 0 1 0 1]
 [0 0 0 0 1 0 1 1]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 1 1 0 0]]


We'll also need the weights and biases for the keys, queries, and values (equations 12.2 and 12.4)

In [3]:
# Choose random values for the parameters
omega = np.random.normal(size=(D,D))
beta = np.random.normal(size=(D,1))
phi = np.random.normal(size=(2*D,1))

We'll need a softmax operation that operates on the columns of the matrix and a ReLU function as well

In [4]:
# Define softmax operation that works independently on each column
def softmax_cols(data_in):
  # Exponentiate all of the values
  exp_values = np.exp(data_in) ;
  # Sum over columns
  denom = np.sum(exp_values, axis = 0);
  # Replicate denominator to N rows
  denom = np.matmul(np.ones((data_in.shape[0],1)), denom[np.newaxis,:])
  # Compute softmax
  softmax = exp_values / denom
  # return the answer
  return softmax


# Define the Rectified Linear Unit (ReLU) function
def ReLU(preactivation):
  activation = preactivation.clip(0.0)
  return activation


In [7]:
 # Now let's compute self attention in matrix form
def graph_attention(X,omega, beta, phi, A):

  # 1. Compute X_prime
  # X_prime = omega @ X + beta (before ReLU application, as ReLU is applied later)
  X_prime = np.matmul(omega, X) + beta

  # 2. Compute S
  # S_ki = phi.T @ [X'_k || X'_i], where k is source node, i is target node
  D = X.shape[0] # Number of dimensions
  N = X.shape[1] # Number of nodes

  phi_left = phi[:D, :]
  phi_right = phi[D:, :]

  # term_k_scores[0, k] = phi_left.T @ X_prime[:, k]
  term_k_scores = np.matmul(phi_left.T, X_prime) # Shape (1, N)

  # term_i_scores[0, i] = phi_right.T @ X_prime[:, i]
  term_i_scores = np.matmul(phi_right.T, X_prime) # Shape (1, N)

  # S[k, i] = term_k_scores[0, k] + term_i_scores[0, i]
  # This computes S where S[row_idx, col_idx] corresponds to attention from row_idx to col_idx
  S = term_k_scores.T + term_i_scores # Shape (N, N)

  # 3. To apply the mask, set S to a very large negative number (e.g. -1e20) everywhere where A+I is zero
  A_plus_I = A + np.eye(N)
  S[A_plus_I == 0] = -1e20

  # 4. Run the softmax function to compute the attention values
  # softmax_cols normalizes each column, so sum_row(attention[row, col]) = 1 for a fixed col.
  # attention[k, i] is the normalized attention from node k to node i.
  attention = softmax_cols(S)

  # 5. Postmultiply X' by the attention values
  # The output for node i is sum_k (attention[k,i] * X_prime[:,k])
  # This is equivalent to X_prime @ attention
  output_pre_relu = np.matmul(X_prime, attention)

  # 6. Apply the ReLU function
  output = ReLU(output_pre_relu)

  return output;

In [6]:
# Test out the graph attention mechanism
np.set_printoptions(precision=3)
output = graph_attention(X, omega, beta, phi, A);
print("Correct answer is:")
print("[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]")
print(" [0.    0.    0.    0.    1.184 0.    2.654 0.  ]")
print(" [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]")
print(" [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]")


print("Your answer is:")
print(output)

Correct answer is:
[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]
 [0.    0.    0.    0.    1.184 0.    2.654 0.  ]
 [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]
 [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]
Your answer is:
[[1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]]


TODO -- Try to construct a dot-product self-attention mechanism as in practical 12.1 that respects the geometry of the graph and has zero attention between non-neighboring nodes by combining figures 13.12a and 13.12b.
